# Flujo – Idealista API → Gold - Feature engeneering y agregación de distancias mínimas a POIs


**Input:**
- `data/processed/idealistaAPI/total_rent_cantabria_outliers.csv`
- `data/processed/idealistaAPI/total_sale_cantabria_outliers.csv`

**Output:**
- `data/gold/final_rent_idealistaAPI.csv`
- `data/gold/final_sale_idealistaAPI.csv`

**Flujo:**
1. Selección y renombrado de columnas útiles del esquema API (48 → ~17)
2. Enriquecimiento con distancias mínimas a POIs (playa, supermercado, colegio)
3. Feature engineering: logs, ratios, interacciones, cuadrados, geo, one-hot
4. Exportación a gold
5. Resumen de features creadas y listado de opcionales

---
## 0. Imports y rutas

In [1]:
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 200)

# ── Raíz del proyecto ──────────────────────────────────────────────────────
def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'data' / 'processed' / 'idealistaAPI').exists():
            return candidate
    raise FileNotFoundError('No se encontró la raíz del proyecto')

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
sys.path.insert(0, str(PROJECT_ROOT))   # necesario para importar src.*

# ── Rutas de entrada ───────────────────────────────────────────────────────
IN_DIR   = PROJECT_ROOT / 'data' / 'processed' / 'idealistaAPI'
RENT_IN  = IN_DIR / 'total_rent_cantabria_outliers.csv'
SALE_IN  = IN_DIR / 'total_sale_cantabria_outliers.csv'

# ── Rutas de salida ────────────────────────────────────────────────────────
GOLD_DIR = PROJECT_ROOT / 'data' / 'gold'
GOLD_DIR.mkdir(parents=True, exist_ok=True)
RENT_OUT = GOLD_DIR / 'final_rent_idealistaAPI.csv'
SALE_OUT = GOLD_DIR / 'final_sale_idealistaAPI.csv'

print(f'Project root : {PROJECT_ROOT}')
print(f'Input  rent  : {RENT_IN}')
print(f'Input  sale  : {SALE_IN}')
print(f'Output rent  : {RENT_OUT}')
print(f'Output sale  : {SALE_OUT}')

MIN_OBS_MUNICIPIO = 10  # municipios con < N obs → colapsan en municipio_otro

Project root : /Users/sitomachucas/Documents/BezanillaSL
Input  rent  : /Users/sitomachucas/Documents/BezanillaSL/data/processed/idealistaAPI/total_rent_cantabria_outliers.csv
Input  sale  : /Users/sitomachucas/Documents/BezanillaSL/data/processed/idealistaAPI/total_sale_cantabria_outliers.csv
Output rent  : /Users/sitomachucas/Documents/BezanillaSL/data/gold/final_rent_idealistaAPI.csv
Output sale  : /Users/sitomachucas/Documents/BezanillaSL/data/gold/final_sale_idealistaAPI.csv


---
## 1. Carga y selección de columnas útiles

De las 48 columnas del esquema API se conservan solo las ~17 con valor analítico.
El resto (IDs, URLs, textos de marketing, flags de display, constantes) se descartan.

In [2]:
# ── Mapeo: nombre API → nombre limpio ─────────────────────────────────────
# Solo se incluyen columnas con valor analítico.
#
# DESCARTADAS (razón):
#   propertyCode, externalReference  → identificadores, no predictivos
#   thumbnail, url                   → URLs
#   numPhotos                        → no usado en ML
#   address                          → texto libre, lat/lon ya lo cubre
#   country                          → constante (España)
#   showAddress                      → flag de display
#   distance                         → artefacto API (distancia al punto de búsqueda)
#   description, notes               → texto no estructurado
#   hasVideo, hasPlan, has3DTour,
#     has360, hasStaging             → no usados en ML
#   status                           → no usado en ML
#   topNewDevelopment,
#     newDevelopmentHighlight,topPlus→ flags de marketing
#   priceInfo.price.amount           → duplicado de price
#   priceInfo.price.currencySuffix   → constante ('€')
#   parkingSpace.isParkingSpaceIncludedInPrice,
#     parkingSpace.parkingSpacePrice → subdetalle no usado en ML
#   suggestedTexts.*, highlight.*    → texto de marketing
#   priceInfo.price.priceDropInfo.*  → >95% nulos
#   newDevelopmentFinished           → no usado en ML
#   source_run_component, source_run → metadatos técnicos
#   propertyType                     → cubierto por detailedType.typology
#   operation                        → constante por dataset

COL_RENAME = {
    'price':                          'precio',
    'size':                           'superficie_construida_m2',
    'rooms':                          'numero_dormitorios',
    'bathrooms':                      'numero_banos',
    'floor':                          'planta',
    'exterior':                       'es_exterior',
    'hasLift':                        'tiene_ascensor',
    'parkingSpace.hasParkingSpace':   'tiene_garaje',
    'newDevelopment':                 'obra_nueva',
    'latitude':                       'latitud',
    'longitude':                      'longitud',
    'municipality':                   'municipio',
    'province':                       'provincia',
    'district':                       'distrito',
    'detailedType.typology':          'tipologia',
    'detailedType.subTypology':       'subtipologia',
    'priceByArea':                    'precio_m2_raw',
}

def load_and_select(path: Path, name: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    print(f'{name} – cargado: {df.shape}')

    rename_map = {k: v for k, v in COL_RENAME.items() if k in df.columns}
    df = df.rename(columns=rename_map)

    keep = [v for v in COL_RENAME.values() if v in df.columns]
    df = df[keep].copy()

    print(f'{name} – tras selección: {df.shape}  |  columnas: {df.columns.tolist()}')
    return df

rent_df = load_and_select(RENT_IN, 'RENT')
sale_df = load_and_select(SALE_IN, 'SALE')

RENT – cargado: (674, 49)
RENT – tras selección: (674, 17)  |  columnas: ['precio', 'superficie_construida_m2', 'numero_dormitorios', 'numero_banos', 'planta', 'es_exterior', 'tiene_ascensor', 'tiene_garaje', 'obra_nueva', 'latitud', 'longitud', 'municipio', 'provincia', 'distrito', 'tipologia', 'subtipologia', 'precio_m2_raw']
SALE – cargado: (2532, 52)
SALE – tras selección: (2532, 17)  |  columnas: ['precio', 'superficie_construida_m2', 'numero_dormitorios', 'numero_banos', 'planta', 'es_exterior', 'tiene_ascensor', 'tiene_garaje', 'obra_nueva', 'latitud', 'longitud', 'municipio', 'provincia', 'distrito', 'tipologia', 'subtipologia', 'precio_m2_raw']


---
## 2. Enriquecimiento con distancias mínimas a POIs

Usa `src/geospatial_expansion` que carga `data/processed/geo/pois_cantabria.csv`
(playas, supermercados y colegios de OSM) y calcula la distancia haversine al POI más cercano.

In [3]:
from src.geospatial_expansion import agregar_distancias_minimas_poi

TIPOS_POI     = ['playa', 'supermercado', 'colegio']
DISTANCE_COLS = [
    'distancia_min_playa_km',
    'distancia_min_supermercado_km',
    'distancia_min_colegio_km',
]

rent_df = agregar_distancias_minimas_poi(rent_df, TIPOS_POI)
sale_df = agregar_distancias_minimas_poi(sale_df, TIPOS_POI)

print('Distancias RENT:')
print(rent_df[DISTANCE_COLS].describe().round(3))
print('\nDistancias SALE:')
print(sale_df[DISTANCE_COLS].describe().round(3))

Distancias RENT:
       distancia_min_playa_km  distancia_min_supermercado_km  \
count                 674.000                        674.000   
mean                    3.216                          0.918   
std                     4.679                          3.280   
min                     0.056                          0.006   
25%                     1.000                          0.108   
50%                     2.104                          0.194   
75%                     2.734                          0.384   
max                    45.219                         35.918   

       distancia_min_colegio_km  
count                   674.000  
mean                      0.919  
std                       3.237  
min                       0.013  
25%                       0.136  
50%                       0.228  
75%                       0.418  
max                      35.777  

Distancias SALE:
       distancia_min_playa_km  distancia_min_supermercado_km  \
count             

---
## 3. Feature Engineering

In [4]:
UNIFAMILIAR_VALUES = {
    'chalet', 'countryhouse', 'singlefamily', 'house',
    'villa', 'townhouse', 'detachedhouse', 'adosado',
}

# ── Helpers ────────────────────────────────────────────────────────────────
def normalize_boolean(series: pd.Series) -> pd.Series:
    true_vals  = {'true', 't', '1', 'yes', 'y', 'si', 's'}
    false_vals = {'false', 'f', '0', 'no', 'n'}
    def _conv(v):
        if pd.isna(v): return 0
        if isinstance(v, (bool, np.bool_)): return int(v)
        if isinstance(v, (int, float, np.integer, np.floating)):
            return 0 if pd.isna(v) else int(v >= 1)
        t = str(v).strip().lower()
        if t in true_vals:  return 1
        if t in false_vals: return 0
        return 0
    return series.map(_conv).astype(int)

def parse_planta(v) -> int:
    if pd.isna(v): return 0
    t = str(v).strip().lower()
    if t in {'bj', 'bajo', 'entresuelo', 'sotano', 'ss', 'st'}: return 0
    m = re.search(r'-?\d+', t)
    return int(m.group()) if m else 0

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

# ── Pipeline principal ─────────────────────────────────────────────────────
def engineer_features(df_input: pd.DataFrame, name: str) -> pd.DataFrame:
    df = df_input.copy()

    # 1. Tipado numérico
    num_cols = [
        'precio', 'superficie_construida_m2', 'numero_dormitorios',
        'numero_banos', 'latitud', 'longitud', 'planta', 'precio_m2_raw',
        *DISTANCE_COLS,
    ]
    for col in num_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # 2. Filtrar precio y superficie > 0
    n0 = len(df)
    df = df[(df['precio'] > 0) & (df['superficie_construida_m2'] > 0)].copy()
    print(f'{name}: {n0} → {len(df)} tras filtro precio/superficie > 0')

    # 3. Imputación por mediana
    for col in ['numero_dormitorios', 'numero_banos', 'latitud', 'longitud', *DISTANCE_COLS]:
        if col in df.columns:
            med = df[col].median()
            df[col] = df[col].fillna(med if not pd.isna(med) else 0.0)

    # 4. Booleanas → 0/1
    for col in ['es_exterior', 'tiene_ascensor', 'tiene_garaje', 'obra_nueva']:
        if col in df.columns:
            df[col] = normalize_boolean(df[col])

    # 5. Tipología unificada + versiones condicionadas a piso (usadas en 51 y 54)
    tip = df.get('tipologia', pd.Series(index=df.index, dtype='string'))
    tip = tip.astype('string').str.lower().str.strip().fillna('')
    df['tipologia_unificada'] = np.where(tip.isin(UNIFAMILIAR_VALUES), 'unifamiliar', 'piso')
    es_piso = (df['tipologia_unificada'] == 'piso').astype(int)
    df['es_exterior_piso']    = df.get('es_exterior',    0) * es_piso
    df['tiene_ascensor_piso'] = df.get('tiene_ascensor', 0) * es_piso

    # 6. Planta numérica
    df['planta_num'] = df['planta'].apply(parse_planta) if 'planta' in df.columns else 0

    # 7. Features logarítmicas
    df['log_precio']                   = np.log(df['precio'])
    df['log_superficie_construida_m2'] = np.log(df['superficie_construida_m2'])

    # 8. Precio por m² y media por municipio
    df['precio_m2']                 = df['precio'] / df['superficie_construida_m2']
    df['precio_m2_municipio_media'] = df.groupby('municipio')['precio_m2'].transform('mean')

    # 9. Ratios (usados en ML)
    df['ratio_dormitorios_superficie'] = df['numero_dormitorios'] / df['superficie_construida_m2']
    df['ratio_banos_superficie']       = df['numero_banos']       / df['superficie_construida_m2']

    # 10. Interacciones (usadas en ML)
    df['interaccion_superficie_banos']         = df['superficie_construida_m2'] * df['numero_banos']
    df['interaccion_planta_sin_ascensor_piso'] = (
        df['planta_num'] * (1 - df.get('tiene_ascensor', 0)) * es_piso
    )

    # 11. Features geográficas
    df['latitud_2']                    = df['latitud']  ** 2
    df['longitud_2']                   = df['longitud'] ** 2
    df['interaccion_latitud_longitud'] = df['latitud']  *  df['longitud']

    centro_lat = df.groupby('municipio')['latitud'].transform('mean')
    centro_lon = df.groupby('municipio')['longitud'].transform('mean')
    df['distancia_centro_municipio_km'] = haversine_km(
        df['latitud'], df['longitud'], centro_lat, centro_lon
    )

    # 12. Score de cercanía a servicios
    # Las inversas individuales son intermedias → se calculan y descartan
    inv_cols = [f'inv_{c}' for c in DISTANCE_COLS]
    for col, inv_col in zip(DISTANCE_COLS, inv_cols):
        df[inv_col] = 1 / (1 + df[col])
    df['score_cercania_servicios'] = df[inv_cols].mean(axis=1)
    df.drop(columns=inv_cols, inplace=True)

    # 13. Términos cuadráticos (usados en ML)
    df['superficie_construida_m2_2'] = df['superficie_construida_m2'] ** 2
    df['numero_banos_2']             = df['numero_banos']       ** 2
    df['numero_dormitorios_2']       = df['numero_dormitorios'] ** 2

    # 14. One-hot: solo tipologia_unificada y municipio (usados en ML)
    #     subtipologia, provincia y distrito se conservan como texto
    for col in ['tipologia_unificada', 'municipio']:
        df[col] = df[col].fillna('desconocido').astype(str).str.strip().replace('', 'desconocido')
    df = pd.get_dummies(df, columns=['tipologia_unificada', 'municipio'],
                        drop_first=False, dtype=int)

    # 15. Colapsar municipios con < MIN_OBS_MUNICIPIO observaciones en 'municipio_otro'
    #     Municipios con muy pocas muestras aportan poco al modelo y generan columnas casi
    #     siempre a 0. Se agrupan en una única variable 'municipio_otro'.
    mun_cols_oh = [c for c in df.columns if c.startswith('municipio_')]
    mun_counts_oh = df[mun_cols_oh].sum()
    rare_muns = mun_counts_oh[mun_counts_oh < MIN_OBS_MUNICIPIO].index.tolist()
    if rare_muns:
        df['municipio_otro'] = df[rare_muns].max(axis=1)
        df = df.drop(columns=rare_muns)
        print(f'  {name}: {len(rare_muns)} municipios → municipio_otro '
              f'(< {MIN_OBS_MUNICIPIO} obs): '
              f"{sorted([m.replace('municipio_','') for m in rare_muns])}")

    # Normalizar bools residuales
    bool_cols = df.select_dtypes(include='bool').columns
    if len(bool_cols):
        df[bool_cols] = df[bool_cols].astype(int)

    print(f'{name}: shape final {df.shape}')
    return df

---
## 4. Aplicar y exportar

In [5]:
rent_final = engineer_features(rent_df, 'RENT')
sale_final = engineer_features(sale_df, 'SALE')

# ── Filtro exacto sobre precio_m2 calculado ─────────────────────────────────
# priceByArea de la API está redondeado a entero: una propiedad con
# priceByArea=18 pasa el outlier notebook pero precio/superficie puede dar
# 18.46. Este filtro aplica el umbral exacto sobre el valor calculado.
n_rent_pre = len(rent_final)
rent_final = rent_final[
    (rent_final["precio_m2"] >= 6) & (rent_final["precio_m2"] <= 18)
].reset_index(drop=True)
n_sale_pre = len(sale_final)
sale_final = sale_final[sale_final["precio_m2"] >= 1000].reset_index(drop=True)
print(f"RENT: {n_rent_pre - len(rent_final)} eliminados por precio_m2 fuera de [6, 18 €/m²]")
print(f"SALE: {n_sale_pre - len(sale_final)} eliminados por precio_m2 < 1000 €/m²")

# Eliminar archivos gold anteriores si existen
for p in [RENT_OUT, SALE_OUT]:
    if p.exists():
        p.unlink()
        print(f'Eliminado: {p.name}')

rent_final.to_csv(RENT_OUT, index=False)
sale_final.to_csv(SALE_OUT, index=False)

print(f'\nExportado: {RENT_OUT}  ({len(rent_final):,} filas, {rent_final.shape[1]} columnas)')
print(f'Exportado: {SALE_OUT}  ({len(sale_final):,} filas, {sale_final.shape[1]} columnas)')

RENT: 674 → 674 tras filtro precio/superficie > 0
  RENT: 49 municipios → municipio_otro (< 10 obs): ['Alfoz de Lloredo', 'Ampuero', 'Barcena de Cicero', 'Beranga', 'Cabezón de la Sal', 'Campoo de Enmedio', 'Cartes', 'Castañeda', 'Colindres', 'Comillas', 'Cudon', 'Entrambasaguas', 'Gallarta', 'Getxo', 'Guarnizo', 'Guriezo', 'Hazas de Cesto', 'Laredo', 'Liendo', 'Limpias', 'Los Corrales de Buelna', 'Marina de Cudeyo', 'Miengo', 'Mogro', 'Molledo', 'Noja', 'Polanco', 'Puente San Miguel', 'Puente Viesgo', 'Ramales de la Victoria', 'Reinosa', 'Reocin', 'Ribadedeva', 'Ribamontan al Mar', 'Ribamontan al Monte', 'San Roque de Riomiera', 'San Vicente de la Barquera', 'Santa Cruz de Bezana', 'Santa Maria de Cayon', 'Santillana del Mar', 'Santoña', 'Santurtzi', 'Solares', 'Suances', 'Udias', 'Val de San Vicente', 'Villapresente', 'Viveda', 'Voto']
RENT: shape final (674, 47)
SALE: 2532 → 2532 tras filtro precio/superficie > 0
  SALE: 30 municipios → municipio_otro (< 10 obs): ['Ajo', 'Alfoz de L

---
## 5. Resumen de features

### 5.1 Features usadas en los notebooks de ML

In [6]:
ML_NOTEBOOKS = {
    '51 OLS base': [
        'log_superficie_construida_m2', 'numero_dormitorios', 'numero_banos',
        'tiene_garaje', 'obra_nueva', 'distancia_min_playa_km',
        'distancia_min_supermercado_km', 'distancia_min_colegio_km',
        'distancia_centro_municipio_km', 'precio_m2_municipio_media',
    ],
    '51 Ridge / Lasso': [
        'superficie_construida_m2', 'numero_dormitorios', 'numero_banos',
        'latitud', 'longitud', 'planta_num', 'es_exterior_piso',
        'tiene_ascensor_piso', 'tiene_garaje', 'obra_nueva',
        'distancia_min_playa_km', 'distancia_min_supermercado_km',
        'distancia_min_colegio_km', 'precio_m2_municipio_media',
        'ratio_dormitorios_superficie', 'ratio_banos_superficie',
        'interaccion_superficie_banos', 'interaccion_planta_sin_ascensor_piso',
        'latitud_2', 'longitud_2', 'interaccion_latitud_longitud',
        'distancia_centro_municipio_km', 'score_cercania_servicios',
        'superficie_construida_m2_2', 'numero_banos_2', 'numero_dormitorios_2',
    ],
    '52 Random Forest / 53 Boosting': [
        'superficie_construida_m2', 'numero_dormitorios', 'numero_banos',
        'tiene_garaje', 'obra_nueva', 'distancia_min_playa_km',
        'distancia_min_supermercado_km', 'distancia_min_colegio_km',
        'distancia_centro_municipio_km', 'score_cercania_servicios',
    ],
    '54 Híbrido': [
        'superficie_construida_m2', 'numero_dormitorios', 'numero_banos',
        'latitud', 'longitud', 'planta_num', 'es_exterior_piso',
        'tiene_ascensor_piso', 'tiene_garaje', 'obra_nueva',
        'distancia_min_playa_km', 'distancia_min_supermercado_km',
        'distancia_min_colegio_km', 'precio_m2_municipio_media',
        'ratio_dormitorios_superficie', 'ratio_banos_superficie',
        'interaccion_planta_sin_ascensor_piso',
        'distancia_centro_municipio_km', 'score_cercania_servicios',
    ],
}

print('Compatibilidad con notebooks de ML (comprobando sobre SALE):')
print('=' * 60)
all_ok = True
for nb_name, required in ML_NOTEBOOKS.items():
    missing = [c for c in required if c not in sale_final.columns]
    status  = '✓' if not missing else '✗'
    print(f'  {status}  {nb_name}')
    if missing:
        print(f'      ⚠ Faltan: {missing}')
        all_ok = False

tip_ok  = any(c.startswith('tipologia_unificada_') for c in sale_final.columns)
mun_ok  = any(c.startswith('municipio_')           for c in sale_final.columns)
log_ok  = 'log_precio' in sale_final.columns
print(f'  {"✓" if tip_ok else "✗"}  One-hot tipologia_unificada_*')
print(f'  {"✓" if mun_ok else "✗"}  One-hot municipio_*')
print(f'  {"✓" if log_ok else "✗"}  log_precio (target en ML)')

if all_ok and tip_ok and mun_ok and log_ok:
    print('\n✅ Gold file compatible con todos los notebooks de ML.')

Compatibilidad con notebooks de ML (comprobando sobre SALE):
  ✓  51 OLS base
  ✓  51 Ridge / Lasso
  ✓  52 Random Forest / 53 Boosting
  ✓  54 Híbrido
  ✓  One-hot tipologia_unificada_*
  ✓  One-hot municipio_*
  ✓  log_precio (target en ML)

✅ Gold file compatible con todos los notebooks de ML.


### 5.2 Features opcionales – no usadas en ningún modelo de ML

Estas columnas están en el gold file pero no aparecen en ningún notebook de ML.  
Puedes eliminarlas ejecutando la celda de abajo.

In [7]:
FEATURES_OPCIONALES = {
    'precio_m2':     'Precio/m² crudo. Solo precio_m2_municipio_media se usa en ML.',
    'precio_m2_raw': 'priceByArea del API. Redundante con precio_m2.',
    'planta':        'Planta en texto crudo. ML usa planta_num (numérico limpio).',
    'tipologia':     'Tipología raw del API. ML usa tipologia_unificada_* (one-hot).',
    'subtipologia':  'Subtipología raw. Sin one-hot, sin uso en ML.',
    'provincia':     'Constante (Cantabria). Sin uso en ML.',
    'distrito':      'Distrito/barrio en texto. Sin one-hot, sin uso en ML.',
    'es_exterior':   'Exterior para todos los tipos. ML usa es_exterior_piso (solo pisos).',
    'tiene_ascensor':'Ascensor para todos los tipos. ML usa tiene_ascensor_piso (solo pisos).',
}

print('=' * 65)
print('FEATURES OPCIONALES EN EL GOLD FILE (no usadas en ML):')
print('=' * 65)
for feat, motivo in FEATURES_OPCIONALES.items():
    presente = '✓' if feat in sale_final.columns else '✗'
    print(f'  [{presente}]  {feat}')
    print(f'             {motivo}')

print('\nPara eliminarlas, descomenta y ejecuta la siguiente celda.')

FEATURES OPCIONALES EN EL GOLD FILE (no usadas en ML):
  [✓]  precio_m2
             Precio/m² crudo. Solo precio_m2_municipio_media se usa en ML.
  [✓]  precio_m2_raw
             priceByArea del API. Redundante con precio_m2.
  [✓]  planta
             Planta en texto crudo. ML usa planta_num (numérico limpio).
  [✓]  tipologia
             Tipología raw del API. ML usa tipologia_unificada_* (one-hot).
  [✓]  subtipologia
             Subtipología raw. Sin one-hot, sin uso en ML.
  [✓]  provincia
             Constante (Cantabria). Sin uso en ML.
  [✓]  distrito
             Distrito/barrio en texto. Sin one-hot, sin uso en ML.
  [✓]  es_exterior
             Exterior para todos los tipos. ML usa es_exterior_piso (solo pisos).
  [✓]  tiene_ascensor
             Ascensor para todos los tipos. ML usa tiene_ascensor_piso (solo pisos).

Para eliminarlas, descomenta y ejecuta la siguiente celda.


In [8]:
# ── Eliminación opcional ───────────────────────────────────────────────────
# Descomenta el bloque y vuelve a ejecutar para eliminar las features opcionales
# del gold file. Los modelos de ML funcionarán igual.

# cols_drop = [c for c in FEATURES_OPCIONALES if c in sale_final.columns]
# rent_final.drop(columns=[c for c in cols_drop if c in rent_final.columns], inplace=True)
# sale_final.drop(columns=cols_drop, inplace=True)
# rent_final.to_csv(RENT_OUT, index=False)
# sale_final.to_csv(SALE_OUT, index=False)
# print(f'Eliminadas: {cols_drop}')
# print(f'RENT → {rent_final.shape} | SALE → {sale_final.shape}')